# 04 — FP&A End-to-End Model

This notebook completes the Connected Planning FP&A engine:
- Revenue (Day 2)
- Opex, COGS, Gross Margin, EBITDA (Day 3)
- **Capex**
- **Depreciation**
- **NOPAT**
- **FCF**
- **NPV by Scenario**


In [ ]:
import xarray as xr
import pandas as pd
import numpy as np

from src.modules.scenario import ScenarioManager
from src.modules.revenue import build_revenue_module
from src.modules.opex import build_opex_module
from src.modules.cogs import build_cogs_module
from src.modules.ebitda import build_ebitda_module
from src.modules.capex import build_capex_module
from src.modules.depreciation import build_depreciation_module
from src.modules.fcf import build_fcf_module, compute_npv

## 1. Load Price & Volume (from Day 2)

In [ ]:
time = pd.period_range('2026Q1', periods=8, freq='Q')
products = ['Core', 'Premium']

price = xr.DataArray(
    np.tile([[45000, 90000],[45500, 91000]], (4,1)),
    dims=("time","product"),
    coords={"time": time, "product": products}
)

volume = xr.DataArray(
    np.tile([[6000,1500],[6050,1525]], (4,1)),
    dims=("time","product"),
    coords={"time": time, "product": products}
)


## 2. Scenario Manager

In [ ]:
scenarios = ['Base','Upside','Downside']
scenario_mgr = ScenarioManager(
    scenarios=scenarios,
    price_multipliers=[1.00,1.02,0.98],
    volume_multipliers=[1.00,1.05,0.95]
)
prices_s = scenario_mgr.apply_price(price)
volumes_s = scenario_mgr.apply_volume(volume)

rev_ds = build_revenue_module(prices_s, volumes_s)
rev_ds

## 3. Opex / COGS / EBITDA (from Day 3)

In [ ]:
base_opex = xr.DataArray(
    np.tile([[22e6,8e6,6e6,10e6]], (len(time),1)),
    dims=("time","cost_category"),
    coords={"time": time, "cost_category": ['People','G&A','IT','Marketing']}
)

inflation = {'Base':1.00, 'Upside':1.03, 'Downside':1.08}
opex_ds = build_opex_module(base_opex, inflation, scenarios)

cogs_ds = build_cogs_module(rev_ds['Revenue'], 0.72)
ebitda_ds = build_ebitda_module(cogs_ds, opex_ds)
ebitda_ds

## 4. Capex & Depreciation

In [ ]:
yearly_capex = {2026:40e6, 2027:28e6}

capex_ds = build_capex_module(yearly_capex, time, scenarios)
dep_ds = build_depreciation_module(capex_ds, useful_life_years=5, time=time)
dep_ds

## 5. Free Cash Flow (FCF)

In [ ]:
nwc_delta = xr.DataArray(
    np.zeros((len(scenarios), len(time))),
    dims=("scenario","time"),
    coords={"scenario":scenarios, "time":time}
)

fcf_ds = build_fcf_module(ebitda_ds, dep_ds, capex_ds, nwc_delta)
fcf_ds

## 6. NPV by Scenario

In [ ]:
npv = compute_npv(fcf_ds['FCF'], discount_rate=0.10)
npv.to_pandas()